# Anomaly Detection using GNN
CERT Insider Threat Dataset - cleaning and Exploratory Data Analysis (EDA) 
Step 1: Preparing Data for Graph and GNN Model Building 

Author: Dagmara Krenich

This file contains the full cleaning and analysis pipeline for all 5 CERT v4.1 dataset files. We use PySpark due to huge data size (over 14 GB).

1. Required libraries

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

2. PySpark Initialization - PySpark allows you to process data larger than available RAM by loading it in batches, without loading the entire thing into memory. "local[*]" mode = all CPU cores.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CERT Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.executorEnv.SPARK_LOCAL_IP", "127.0.0.1") \
    .getOrCreate()


## Section A: Helpers
A set of functions that will be used for each CERT data file. 

In [21]:
def code_quality(df, name):
    '''
    Displays a basic quality report for the data frame:
    - number of records
    - number of columns
    - number and percentage of missing values ​​in each column
    - number of duplicates
    '''
    print(f"Data quality for: {name}")

    n_rows = df.count()
    n_cols = len(df.columns)
    print(f"Number of records: {n_rows}, number of columns: {n_cols}")

    # Missing values - for each column if null or "" -> 1, else 0, return sum
    missing_exprs = [
        F.sum(
            F.when(
                F.col(c).isNull() | (F.col(c) == ""), 1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    missing_df = df.select(missing_exprs)
    print("Missing df:", missing_df.collect())
    for row in missing_df.collect():
        for col_name in df.columns:
            nulls = row[col_name]
            if nulls > 0:
                pct = 100 * nulls / n_rows
                print(f"{col_name:<25} {nulls:>8,} ({pct:.2f}%)")

    # Duplicates
    n_distinct = df.dropDuplicates().count()
    n_dup = n_rows - n_distinct

    print(f"\nDuplicates: {n_dup:,} ({100*n_dup/n_rows:.2f}%)")

    return {
        "rows": n_rows,
        "columns": n_cols,
        "duplicates": n_dup,
        "missing": missing_df
    }

In [34]:
def parse_date(df, column="date"):
    """
    Converts date type columns to Timestamp
    Format that CERT uses: MM/DD/YYYY HH:MM:SS
    Returns timestamp column and datetime slices as year, month, day, hour, day_of_the_week
    """
    df = df.withColumn(
        "timestamp",
        F.to_timestamp(F.col(column), "MM/dd/yyyy HH:mm:ss")
    )
    df = df.withColumn("year", F.year("timestamp"))
    df = df.withColumn("month", F.month("timestamp"))
    df = df.withColumn("day", F.day("timestamp"))
    df = df.withColumn("hour", F.hour("timestamp"))
    # day of week: 1 - Sunday 
    df = df.withColumn("day_of_the_week", F.dayofweek("timestamp"))
    # Flag - whether the incident happened during of after business hours
    df = df.withColumn("not_working_hours", (F.col("hour") < 8) | (F.col("hour") > 18 ))
    return df

In [35]:
def letters_cleaning(df):
    # Convert column names to lower case
    new_columns = [c.lower() for c in df.columns]
    return df.toDF(*new_columns)

## Section B: logon.csv 

In [36]:
df_logon = spark.read.csv("../data/logon.csv", header=True, inferSchema=True)

File: "logon.csv" contains login and logout events by users to the computers. <br>
Columns:
- id (generic id)
- date
- user 
- pc
- activity (login, logoff)

Extended meaning for GNN: user->pc edge, activity patterns over time

In [37]:
df_logon.show(3, truncate=False)

+------------------------+-------------------+-------+-------+--------+
|id                      |date               |user   |pc     |activity|
+------------------------+-------------------+-------+-------+--------+
|{X0W9-Q2DW16EI-1074QDVQ}|01/02/2010 05:02:50|WCR0044|PC-9174|Logon   |
|{C2O4-Z2RH12FQ-9176MUEL}|01/02/2010 05:19:09|WCR0044|PC-9174|Logoff  |
|{U1J8-P4HX02EV-5327GONH}|01/02/2010 06:22:11|WCR0044|PC-5494|Logon   |
+------------------------+-------------------+-------+-------+--------+
only showing top 3 rows


In [38]:
df_logon.printSchema()

root
 |-- id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- user: string (nullable = true)
 |-- pc: string (nullable = true)
 |-- activity: string (nullable = true)



In [39]:
# Quality Data Check
logon_report = code_quality(df_logon, "logon.csv")

Data quality for: logon.csv
Number of records: 899118, number of columns: 5
Missing df: [Row(id=0, date=0, user=0, pc=0, activity=0)]

Duplicates: 0 (0.00%)


In [40]:
# Cleaning logon data 
logon = letters_cleaning(df_logon)
logon = logon \
    .dropDuplicates() \
    .filter(F.col("user").isNotNull()) \
    .filter(F.col("pc").isNotNull()) \
    .filter(F.col("activity").isin("Logon", "Logoff"))

In [41]:
# Logon date parse
logon = parse_date(logon)

In [42]:
logon.show(3, truncate=False)

+------------------------+-------------------+-------+-------+--------+-------------------+----+-----+---+----+---------------+-----------------+
|id                      |date               |user   |pc     |activity|timestamp          |year|month|day|hour|day_of_the_week|not_working_hours|
+------------------------+-------------------+-------+-------+--------+-------------------+----+-----+---+----+---------------+-----------------+
|{Q2G2-D4OC17ON-2211OKVU}|01/04/2010 07:31:00|NCT0467|PC-5648|Logon   |2010-01-04 07:31:00|2010|1    |4  |7   |2              |true             |
|{T5G4-C2ZX21KH-5381BTIG}|01/04/2010 13:29:00|BFF0257|PC-0245|Logon   |2010-01-04 13:29:00|2010|1    |4  |13  |2              |false            |
|{I6T5-X0PO79WB-0037HSZP}|01/04/2010 17:03:00|STD0242|PC-2487|Logoff  |2010-01-04 17:03:00|2010|1    |4  |17  |2              |false            |
+------------------------+-------------------+-------+-------+--------+-------------------+----+-----+---+----+-------------